<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/0/0f/We_logo.svg/3840px-We_logo.svg.png" alt="WE Logo" width="60" style="margin-bottom:8px"/>

# Telecom Egypt (WE) — ML Project
## Smart Plan Recommendation System

---

### Background

Telecom Egypt (WE) serves thousands of subscribers across Egypt with internet plans that vary in speed, quota, and price. A key business challenge is **plan-subscriber mismatch** — subscribers enrolled in plans that do not reflect their actual usage and payment behavior.

This project simulates a real-world data science task at WE. You are part of the Data & AI team. Your mission is to build a machine learning model that analyzes subscriber profiles and recommends the most suitable plan action:

> **Upgrade** — the subscriber needs a bigger plan  
> **Downgrade** — the subscriber is over-paying for what they use  
> **Keep** — the current plan is a good fit

---

### Available Datasets

| Dataset | Description |
|---------|-------------|
| `Customer.csv` | Subscriber demographics and plan enrollment |
| `Subscription_Plan_Lkp.csv` | Plan details — quota, speed, and price |
| `Network_Elements.csv` | Infrastructure type per subscriber (VDSL, FTTH, etc.) |
| `Payments.csv` | Monthly payment transactions per subscriber |
| `Consumption_RG_LKP.csv` | Lookup table for consumption rating groups |

---

### General Instructions

- Read each step carefully before writing any code.
- Each step has a clear objective — understand **why** you are doing it, not just **how**.
- You are expected to make decisions along the way. Document your reasoning in markdown cells.
- There is no single correct answer — justify your choices.


---
## Step 1 — Load & Explore the Data

### Objective
Understand what data you have before touching it.

### Why This Step Matters
Before building any model, a data scientist must fully understand the data — its structure, quality, and distributions. Skipping EDA leads to wrong assumptions and poor models.

### What To Do
- Load all 5 datasets and print their shape and data types.
- Check for missing values in each table and decide how to handle them.
- Visualize key distributions:
  - `Customer`: `SUBSCRIBER_STATUS`, `CUSTOMER_CLASS`, `GENDER`
  - `Subscription_Plan_Lkp`: price range, speed range, quota sizes
  - `Payments`: revenue columns and monthly revenue trend
  - `Network_Elements`: breakdown of `TECHNOLOGY_TYPE`

**Expected Output:** At least 4 visualizations. Write a short observation below each one explaining what you see.


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", None)


In [ ]:
# ── Load Data ────────────────────────────────────────────────────────────────
DATA_PATH = "Data For Task 2/"   # adjust if your files are elsewhere

customer = pd.read_csv(DATA_PATH + "Customer.csv")
plans    = pd.read_csv(DATA_PATH + "Subscription_Plan_Lkp.csv")
network  = pd.read_csv(DATA_PATH + "Network_Elements.csv")
payments = pd.read_csv(DATA_PATH + "Payments.csv")
rg_lkp   = pd.read_csv(DATA_PATH + "Consumption_RG_LKP.csv")

for name, df in [("Customer", customer), ("Plans", plans), ("Network", network),
                 ("Payments", payments), ("Consumption_RG_LKP", rg_lkp)]:
    print(f"{name:<20} shape: {df.shape}")
    print(df.dtypes)
    print("-" * 60)


In [ ]:
# ── Missing Values ────────────────────────────────────────────────────────────
for name, df in [("Customer", customer), ("Plans", plans), ("Network", network),
                 ("Payments", payments), ("Consumption_RG_LKP", rg_lkp)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print(f"--- {name} ---")
    print(nulls if len(nulls) else "No missing values")
    print()

# Handling decisions:
# - Customer.GROUP_ID#  -> ~93% missing, not useful as a feature -> drop the column
# - Customer.BIRTHDATE  -> few missing, we don't use age directly -> leave as is
customer = customer.drop(columns=["GROUP_ID#"])
print("Dropped GROUP_ID# from Customer.")


In [ ]:
# ── EDA: Customer Table ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

customer["SUBSCRIBER_STATUS"].value_counts().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Subscriber Status Distribution")
axes[0].set_ylabel("Count")

customer["CUSTOMER_CLASS"].value_counts().plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].set_title("Customer Class Distribution")
axes[1].set_ylabel("Count")

customer["GENDER"].value_counts().plot(kind="pie", ax=axes[2], autopct="%1.1f%%")
axes[2].set_title("Gender Distribution")
axes[2].set_ylabel("")

plt.tight_layout()
plt.show()

# Observation:
# The vast majority of subscribers are Active. Customer classes are dominated by
# a few segments (e.g. B+), and gender split is roughly balanced.


In [ ]:
# ── EDA: Plans Table ─────────────────────────────────────────────────────────
# QUOTA is stored as text (contains values like "1TB") -> convert to numeric GB
def quota_to_gb(q):
    q = str(q).strip().upper()
    if q.endswith("TB"):
        return float(q.replace("TB", "")) * 1024
    return pd.to_numeric(q, errors="coerce")

plans["QUOTA_NUM"] = plans["QUOTA"].apply(quota_to_gb)

active_plans = plans[plans["PRICE_PLAN_PRICE"] > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(active_plans["PRICE_PLAN_PRICE"], bins=20, color="seagreen", edgecolor="black")
axes[0].set_title("Distribution of Plan Prices (EGP)")
axes[0].set_xlabel("Price (EGP)")
axes[0].set_ylabel("Number of Plans")

axes[1].hist(active_plans["QUOTA_NUM"], bins=20, color="indianred", edgecolor="black")
axes[1].set_title("Distribution of Plan Quotas (GB)")
axes[1].set_xlabel("Quota (GB)")
axes[1].set_ylabel("Number of Plans")

plt.tight_layout()
plt.show()

# Observation:
# Most plans are priced in the lower range with a long tail of premium plans.
# Quotas cluster around a few common sizes, with rare very large (1TB+) plans.


In [ ]:
# ── EDA: Payments Table ──────────────────────────────────────────────────────
revenue_cols = ["RENT_REVENUE", "OUT_BUNDLE_REVENUE", "CREATION_FEES_REVENUE",
                "DEVICES_REVENUE", "IN_BUNDLE_REVENUE", "ADDON_REVENUE"]
payments["total_payment"] = payments[revenue_cols].sum(axis=1)
payments["CONNECT_DATE"] = pd.to_datetime(payments["CONNECT_DATE"])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Total revenue per subscriber
rev_per_sub = payments.groupby("SUBSCRIBER_ID#")["total_payment"].sum()
axes[0].hist(rev_per_sub, bins=50, color="slateblue", edgecolor="black")
axes[0].set_title("Total Revenue per Subscriber")
axes[0].set_xlabel("Total Payment (EGP)")
axes[0].set_ylabel("Number of Subscribers")

# Monthly revenue trend
monthly_rev = payments.groupby(payments["CONNECT_DATE"].dt.to_period("M"))["total_payment"].sum()
monthly_rev.index = monthly_rev.index.to_timestamp()
axes[1].plot(monthly_rev.index, monthly_rev.values, color="crimson")
axes[1].set_title("Monthly Revenue Trend")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Total Revenue (EGP)")

plt.tight_layout()
plt.show()

# Bonus: Network technology breakdown
network["TECHNOLOGY_TYPE"].value_counts().plot(kind="bar", color="teal", figsize=(8, 3))
plt.title("Network Technology Type Breakdown")
plt.ylabel("Count")
plt.show()

# Observation:
# Subscriber revenue is right-skewed: most subscribers generate modest revenue
# while a small group generates much more. Monthly revenue is fairly stable
# over the period. Fiber/VDSL technologies dominate the network.


---
## Step 2 — Join the Tables

### Objective
Build a single unified subscriber profile by merging all relevant tables.

### Why This Step Matters
Real-world data is never in one place. The ability to correctly join tables — using the right keys and avoiding duplicates or row explosions — is a core data engineering skill.

### What To Do
- Start from `Customer.csv` as the base table.
- Join `Subscription_Plan_Lkp` on `SUBSCRIPTION_PLAN_ID#` → adds quota, speed, price.
- Join `Network_Elements` on `SERVICE_NUMBER#` → adds technology type.
- Aggregate `Payments` per subscriber, then join on `SUBSCRIBER_ID#`.
- After each join, verify the row count has not changed unexpectedly.

> **Watch out for:** Duplicate plan IDs in the lookup table — handle them before joining to avoid inflating your row count.


In [ ]:
# ── Clean Plans ───────────────────────────────────
# The lookup may contain duplicate plan IDs -> keep one row per plan ID
before = len(plans)
plans_clean = plans.drop_duplicates(subset="SUBSCRIPTION_PLAN_ID#", keep="first")
print(f"Plans rows: {before} -> {len(plans_clean)} (removed {before - len(plans_clean)} duplicates)")

plan_cols = ["SUBSCRIPTION_PLAN_ID#", "SUBSCRIPTION_PLAN_DESC", "QUOTA_NUM", "SPEED", "PRICE_PLAN_PRICE"]
plans_clean = plans_clean[plan_cols]


In [ ]:
# ── Join Plans ────────────────────────────────────────────────────────────────
print("Customer rows before join:", len(customer))
df = customer.merge(plans_clean, on="SUBSCRIPTION_PLAN_ID#", how="left")
print("Rows after joining Plans :", len(df))
assert len(df) == len(customer), "Row explosion detected after plans join!"


In [ ]:
# ── Join Network ──────────────────────────────────────────────────────────────
net_cols = ["SERVICE_NUMBER#", "TECHNOLOGY_TYPE", "GOV"]
df = df.merge(network[net_cols].drop_duplicates(subset="SERVICE_NUMBER#"),
              on="SERVICE_NUMBER#", how="left")
print("Rows after joining Network:", len(df))
assert len(df) == len(customer), "Row explosion detected after network join!"


In [ ]:
# ── Aggregate Payments ────────────────────────────────────────────────────────
payments["month"] = payments["CONNECT_DATE"].dt.to_period("M")

pay_agg = payments.groupby("SUBSCRIBER_ID#").agg(
    total_revenue=("total_payment", "sum"),
    addon_total=("ADDON_REVENUE", "sum"),
    out_bundle_total=("OUT_BUNDLE_REVENUE", "sum"),
    n_payments=("Payment_ID", "count"),
    n_months=("month", "nunique"),
).reset_index()

df = df.merge(pay_agg, on="SUBSCRIBER_ID#", how="left")
print("Rows after joining Payments:", len(df))
assert len(df) == len(customer), "Row explosion detected after payments join!"
df.head()


---
## Step 3 — Feature Engineering

### Objective
Transform raw columns into meaningful features that help the model learn patterns.

### Why This Step Matters
Raw data rarely goes directly into a model. Feature engineering is where domain knowledge meets data science — the better your features, the better your model.

### Features To Create

| Feature | Formula | What It Captures |
|---------|---------|-----------------|
| `tenure_days` | `reference_date − ACTIVATION_DATE` | How long the subscriber has been active |
| `avg_monthly_rev` | Mean of payments per subscriber | Average spending per month |
| `addon_ratio` | `addon_total / total_revenue` | How often the subscriber exceeds their bundle |
| `price_to_quota_ratio` | `PRICE_PLAN_PRICE / QUOTA_NUM` | Cost per GB — how expensive the plan is relative to its size |

---

**Formula Details:**

**addon_ratio:**

$$\text{addon_ratio} = \frac{\text{ADDON_REVENUE}}{\text{total_revenue}}$$

*Example:* Subscriber paid 500 EGP total, 100 EGP was addon → addon_ratio = 100/500 = **0.20** (20% outside bundle)

---

**price_to_quota_ratio:**

$$\text{price_to_quota_ratio} = \frac{\text{PRICE_PLAN_PRICE}}{\text{QUOTA_NUM}}$$

*Example:* Plan costs 570 EGP for 400 GB → price_to_quota_ratio = 570/400 = **1.43 EGP per GB**

---

After creating each feature, print a `.describe()` and check for outliers or unexpected values.


In [ ]:
# ── tenure_days ──────────────────────────────────────────────────────────────
df["ACTIVATION_DATE"] = pd.to_datetime(df["ACTIVATION_DATE"])
reference_date = payments["CONNECT_DATE"].max()   # last date in the payments data
df["tenure_days"] = (reference_date - df["ACTIVATION_DATE"]).dt.days
print("Reference date:", reference_date.date())
df["tenure_days"].describe()


In [ ]:
# ── avg_monthly_rev & addon_ratio ────────────────────────────────────────────
# avg_monthly_rev = total revenue / number of months with payments
df["avg_monthly_rev"] = df["total_revenue"] / df["n_months"]

# addon_ratio = ADDON_REVENUE / total_revenue
df["addon_ratio"] = np.where(df["total_revenue"] > 0,
                             df["addon_total"] / df["total_revenue"], 0)
df[["avg_monthly_rev", "addon_ratio"]].describe()


In [ ]:
# ── price_to_quota_ratio ─────────────────────────────────────────────────────
# price_to_quota_ratio = PRICE_PLAN_PRICE / QUOTA_NUM (EGP per GB)
df["price_to_quota_ratio"] = np.where(df["QUOTA_NUM"] > 0,
                                      df["PRICE_PLAN_PRICE"] / df["QUOTA_NUM"], np.nan)
df["price_to_quota_ratio"].describe()


In [ ]:
# ── Summary: All Engineered Features ─────────────────────────────────────────
features_summary = df[["tenure_days", "avg_monthly_rev", "addon_ratio", "price_to_quota_ratio"]]
features_summary.describe().round(3)


---
## Step 4 — Build the Target Variable

### Objective
Define the label the model will learn to predict.

### Why This Step Matters
In supervised ML, the quality of your label directly determines the quality of your model. Here, there is no pre-labeled column — you must define the business logic yourself.

### Labeling Logic

First, compute the **pay_ratio** — how much the subscriber actually pays relative to their plan price:

$$\text{pay_ratio} = \frac{\text{avg_monthly_rev}}{\text{PRICE_PLAN_PRICE}}$$

Then apply the following rules:

| Condition | Label |
|-----------|-------|
| `addon_ratio > 0.15` OR `pay_ratio > 1.30` | `Upgrade` — paying beyond their plan |
| `pay_ratio < 0.50` | `Downgrade` — consistently under-paying |
| Everything else | `Keep` — plan is a good fit |

**Examples:**

| Subscriber | Plan Price | Avg Monthly Pay | pay_ratio | addon_ratio | Label |
|-----------|-----------|----------------|-----------|-------------|-------|
| A | 400 EGP | 550 EGP | 1.375 → 137% | 0.05 | **Upgrade** |
| B | 400 EGP | 180 EGP | 0.450 → 45% | 0.01 | **Downgrade** |
| C | 400 EGP | 350 EGP | 0.875 → 87% | 0.08 | **Keep** |


---

After labeling: print the class distribution. If any class has fewer than 5% of records, discuss in a markdown cell whether this is a problem and how you would handle it.


In [ ]:
# ── Compute pay_ratio & Build Label ──────────────────────────────────────────
df["pay_ratio"] = np.where(df["PRICE_PLAN_PRICE"] > 0,
                           df["avg_monthly_rev"] / df["PRICE_PLAN_PRICE"], np.nan)

def label_subscriber(row):
    if pd.isna(row["pay_ratio"]):
        return np.nan
    if (row["addon_ratio"] > 0.15) or (row["pay_ratio"] > 1.30):
        return "Upgrade"
    if row["pay_ratio"] < 0.50:
        return "Downgrade"
    return "Keep"

df["label"] = df.apply(label_subscriber, axis=1)
df[["pay_ratio", "addon_ratio", "label"]].head(10)


In [ ]:
# ── Class Distribution ────────────────────────────────────────────────────────
print(df["label"].value_counts())
print()
print((df["label"].value_counts(normalize=True) * 100).round(2).astype(str) + " %")

df["label"].value_counts().plot(kind="bar", color=["seagreen", "steelblue", "indianred"], figsize=(7, 3))
plt.title("Class Distribution — Plan Recommendation Labels")
plt.ylabel("Number of Subscribers")
plt.show()


---
## Step 5 — Train & Compare Models

### Objective
Train multiple classifiers and understand the tradeoffs between them.

### Why This Step Matters
No single model is always best. A good data scientist knows when to use a simple interpretable model versus a complex one — and can justify that choice to a non-technical business stakeholder.

### Models To Train

| Model | Why |
|-------|-----|
| **Logistic Regression** | Your baseline — fast, interpretable, good for linear relationships |
| **Random Forest** | Handles non-linearity and feature interactions well |
| **Gradient Boosting** | Typically stronger performance but slower to train |

### Rules
- Use the same train/test/val split for all models `random_state=42`
- Print the Classification Report for each model
- Note any class where the model performs poorly and explain why in a markdown cell


In [ ]:
# ── Select Features & Filter ─────────────────────────────────────────────────
feature_cols = [
    "tenure_days", "avg_monthly_rev", "addon_ratio", "price_to_quota_ratio",
    "QUOTA_NUM", "SPEED", "PRICE_PLAN_PRICE",
    "CUSTOMER_CLASS", "GENDER", "SUBSCRIBER_TYPE", "TECHNOLOGY_TYPE",
]

# Keep only rows that have a label (subscribers with payments and a priced plan)
model_df = df.dropna(subset=["label"]).copy()
print("Rows used for modeling:", len(model_df), "out of", len(df))
model_df[feature_cols + ["label"]].head()


In [ ]:
# ── Handle Missing Values ─────────────────────────────────────────────────────
print("Missing values before handling:")
print(model_df[feature_cols].isnull().sum()[model_df[feature_cols].isnull().sum() > 0])

# Numeric -> fill with median | Categorical -> fill with mode
numeric_feats = ["tenure_days", "avg_monthly_rev", "addon_ratio",
                 "price_to_quota_ratio", "QUOTA_NUM", "SPEED", "PRICE_PLAN_PRICE"]
categorical_feats = ["CUSTOMER_CLASS", "GENDER", "SUBSCRIBER_TYPE", "TECHNOLOGY_TYPE"]

for col in numeric_feats:
    model_df[col] = model_df[col].fillna(model_df[col].median())
for col in categorical_feats:
    model_df[col] = model_df[col].fillna(model_df[col].mode()[0])

print("\nMissing values after handling:", model_df[feature_cols].isnull().sum().sum())


In [ ]:
# ── Encode Categorical Features ──────────────────────────────────────────────
encoded_df = pd.get_dummies(model_df[feature_cols], columns=categorical_feats, drop_first=True)

le = LabelEncoder()
y = le.fit_transform(model_df["label"])
print("Classes:", list(le.classes_))
print("Feature matrix shape:", encoded_df.shape)


In [ ]:
# ── Train / Test Split ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    encoded_df, y, test_size=0.20, random_state=42, stratify=y
)

# Scale features (needed for Logistic Regression, harmless for trees)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train.shape, "| Test:", X_test.shape)


In [ ]:
# ── Model 1: Logistic Regression (Baseline) ──────────────────────────────────
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)

print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))


In [ ]:
# ── Model 2: Random Forest ───────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))


In [ ]:
# ── Model 3: Gradient Boosting ───────────────────────────────────────────────
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

print("=== Gradient Boosting ===")
print(classification_report(y_test, y_pred_gb, target_names=le.classes_))


**Observation on model performance:**

All three models score very high because the label itself was built from engineered features (`addon_ratio`, `pay_ratio` derived from `avg_monthly_rev` and `PRICE_PLAN_PRICE`), so the models can almost perfectly reconstruct the labeling rules.

The weakest class for every model is **Downgrade** — it has very few samples (the data is heavily imbalanced toward Upgrade), so the models see too few examples of it during training. Logistic Regression suffers the most since it is a linear model and cannot capture the threshold-based rules as well as the tree ensembles.


---
## Step 6 — Evaluate

### Objective
Measure model performance beyond just accuracy.


In [ ]:
# ── Model Comparison ─────────────────────────────────────────────────────────
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "Gradient Boosting"],
    "Accuracy": [accuracy_score(y_test, y_pred_lr),
                 accuracy_score(y_test, y_pred_rf),
                 accuracy_score(y_test, y_pred_gb)],
    "Macro F1": [f1_score(y_test, y_pred_lr, average="macro"),
                 f1_score(y_test, y_pred_rf, average="macro"),
                 f1_score(y_test, y_pred_gb, average="macro")],
}).round(4)

print(results.to_string(index=False))

best_idx = results["Macro F1"].idxmax()
best_name = results.loc[best_idx, "Model"]
best_pred = [y_pred_lr, y_pred_rf, y_pred_gb][best_idx]
print(f"\nBest model: {best_name}")


In [ ]:
# ── Confusion Matrix (Best Model) ────────────────────────────────────────────
cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f"Confusion Matrix — {best_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


---
## Step 7 — Business Impact

### Objective
Translate model predictions into actionable business recommendations.



In [ ]:
# ── Predictions on Full Dataset ──────────────────────────────────────────────
best_model = [log_reg, rf, gb][best_idx]

if best_name == "Logistic Regression":
    full_pred = best_model.predict(scaler.transform(encoded_df))
else:
    full_pred = best_model.predict(encoded_df)

model_df["predicted_action"] = le.inverse_transform(full_pred)
print(model_df["predicted_action"].value_counts())


In [ ]:
# ── Revenue Opportunity ──────────────────────────────────────────────────────
# Upgrade candidates: revenue we could capture by moving them to a plan that
# matches what they actually pay (avg monthly pay above current plan price).
upgrades = model_df[model_df["predicted_action"] == "Upgrade"]
upgrade_gap = (upgrades["avg_monthly_rev"] - upgrades["PRICE_PLAN_PRICE"]).clip(lower=0)

# Downgrade candidates: revenue currently at churn risk (over-paying customers).
downgrades = model_df[model_df["predicted_action"] == "Downgrade"]
downgrade_gap = (downgrades["PRICE_PLAN_PRICE"] - downgrades["avg_monthly_rev"]).clip(lower=0)

print(f"Upgrade candidates   : {len(upgrades):,}")
print(f"  Potential extra monthly revenue if upgraded : {upgrade_gap.sum():,.0f} EGP")
print()
print(f"Downgrade candidates : {len(downgrades):,}")
print(f"  Monthly over-payment (churn risk)           : {downgrade_gap.sum():,.0f} EGP")
print()
print("Business takeaway: proactively upgrading heavy users captures revenue they")
print("already spend on add-ons, while right-sizing over-paying customers reduces churn.")


In [ ]:
# ── Upgrade Candidates Profile ───────────────────────────────────────────────
profile = upgrades[["tenure_days", "avg_monthly_rev", "addon_ratio",
                    "PRICE_PLAN_PRICE", "QUOTA_NUM"]].describe().round(2)
print("--- Upgrade Candidates: Numeric Profile ---")
print(profile)

print("\n--- Top Customer Classes among Upgrade Candidates ---")
print(upgrades["CUSTOMER_CLASS"].value_counts().head())

print("\n--- Technology Type among Upgrade Candidates ---")
print(upgrades["TECHNOLOGY_TYPE"].value_counts())
